# Steam game recommendation (GPT) — same methodology as Steam_Mistral

1. **Data**: Run `python steam_data_prep.py` first. Load Steam prep: 2 few-shot users (10 history + 10 recommendations each), 100 target users, sample_for_prompt.
2. **Prompt types**: zero_shot, few_shot, zero_shot_cot, few_shot_cot.
3. **Templates**: A, B, C (JSON with ranks / JSON titles only / plain-text numbered list).
4. **Runs**: First at **temp=0** (normal_t0). Then **varying temperatures** and **self-consistent CoT**.

## 1. Load Steam prep data (run steam_data_prep.py first)

In [ ]:
import json
import pandas as pd
from pathlib import Path

PREP_DIR = Path("Data/steam_prep")

with open(PREP_DIR / "few_shot_users.json") as f:
    few_shot_users = json.load(f)
with open(PREP_DIR / "target_users.json") as f:
    target_users = json.load(f)
with open(PREP_DIR / "sample_for_prompt.json", encoding="utf-8") as f:
    sample_for_prompt = json.load(f)
with open(PREP_DIR / "examples_steam.json", encoding="utf-8") as f:
    EXAMPLES_DATA = json.load(f)

# User history from prep: alternating order (top1, least1, top2, least2, ...). Preserve order when building prompts.
# df_filtered_user_data: user_id + sample_random (list of (game, hours) in prep order)
records = []
for uid in target_users:
    sample = sample_for_prompt.get(uid, [])
    pairs = [(x["game"], x["hours"]) for x in sample if x.get("game")]
    records.append({"user_id": uid, "sample_random": pairs})
df_filtered_user_data = pd.DataFrame(records)

EXAMPLE_USERS = few_shot_users
print("Few-shot users:", few_shot_users)
print("Target users:", len(target_users))
print("df_filtered_user_data shape:", df_filtered_user_data.shape)
print("Examples (10 history + 10 recs per user):", len(EXAMPLES_DATA.get("random", [])))

## 2. OpenAI config + parameters

In [ ]:
import os, re, time, random, json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
from tqdm import tqdm

from openai import OpenAI

#API_KEY = ""  # <-- fill with your OpenAI API key
#MODEL_NAME = "gpt-4o-mini"
#client = OpenAI(api_key=API_KEY)

TOP_K = 10
NORMAL_TEMPS = [0.0]
VARIING_TEMPS = [0.2, 0.5, 0.7, 1.0, 1.2, 1.4, 1.6]  # GPT: 7 temps (includes 1.2, 1.4, 1.6)
NUM_SAMPLES_VARYING_TEMP = 20  # 20 samples for self-consistency (zero_shot_cot, few_shot_cot only)
STYLES_20_SAMPLES = ("zero_shot_cot", "few_shot_cot")  # self-consistency: 20 samples for these two
MAX_WORKERS = 2
MAX_WORKERS_VARYING_TEMP = 8  # more parallelism for varying-temp run (faster)
PAUSE_SECONDS = 0.7

OUTPUT_NORMAL = Path("Data/steam_gpt_results")
OUTPUT_NORMAL.mkdir(parents=True, exist_ok=True)

## 3. Prompt templates A, B, C (games + hours) — zero_shot, few_shot, zero_shot_cot, few_shot_cot

In [ ]:
PROMPT_TEMPLATES_A = {
    "zero_shot": {
        "system": (
            "You are a helpful game recommender.\\n"
            "Recommend exactly the top-{k} ranked games.\\n"
            "RULES (STRICT):\\n"
            "- Recommend {k} distinct games not in the played list below.\\n"
            "- No duplicates.\\n"
            "- Output a single JSON object and nothing else.\\n"
            "SCHEMA (STRICT):\n"
            "{{\"k\": {k}, \"recommendations\": [{{\"rank\": 1, \"title\": \"<Game 1>\"}}, ..., "
            "{{\"rank\": {k}, \"title\": \"<Game {k}>\"}}]}}\n"
        ),
        "user": "User's played games (game name and hours played; exclude all of these):\\n{history}"
    },
    "few_shot": {
        "system": (
            "You are a helpful game recommender.\\n"
            "Below are demonstrated examples (no reasoning). Each example shows a user's played games (game name and hours played) and a FINAL_JSON.\\n"
            "Recommend exactly the top-{k} ranked games.\\n"
            "RULES (STRICT):\\n"
            "- Recommend {k} distinct games not in the played list below.\\n"
            "- No duplicates. Output a single JSON.\\n"
            "SCHEMA (STRICT):\n"
            "{{\"k\": {k}, \"recommendations\": [{{\"rank\": 1, \"title\": \"<Game 1>\"}}, ..., "
            "{{\"rank\": {k}, \"title\": \"<Game {k}>\"}}]}}\n"
        ),
        "user": "User's played games (game name and hours played; exclude all of these):\\n{history}"
    },
    "zero_shot_cot": {
        "system": (
            "You are a helpful game recommender.\\n"
            "Recommend exactly the top-{k} ranked games.\\n"
            "RULES (STRICT):\\n"
            "- Recommend {k} distinct games not in the played list below.\\n"
            "- No duplicates.\\n"
            "- Output a single JSON object and nothing else.\\n"
            "SCHEMA (STRICT):\n"
            "{{\"k\": {k}, \"recommendations\": [{{\"rank\": 1, \"title\": \"<Game 1>\"}}, ..., "
            "{{\"rank\": {k}, \"title\": \"<Game {k}>\"}}]}}\n"
            "Before answering, LET'S THINK STEP BY STEP.\\n"
        ),
        "user": "User's played games (game name and hours played; exclude all of these):\\n{history}"
    },
    "few_shot_cot": {
        "system": (
            "You are a helpful game recommender.\\n"
            "Below are demonstrated examples (no reasoning). Each example shows a user's played games (game name and hours played) and a FINAL_JSON.\\n"
            "Recommend exactly the top-{k} ranked games.\\n"
            "RULES (STRICT):\\n"
            "- Recommend {k} distinct games not in the played list below.\\n"
            "- No duplicates. Output a single JSON.\\n"
            "SCHEMA (STRICT):\n"
            "{{\"k\": {k}, \"recommendations\": [{{\"rank\": 1, \"title\": \"<Game 1>\"}}, ..., "
            "{{\"rank\": {k}, \"title\": \"<Game {k}>\"}}]}}\n"
            "Before answering, LET'S THINK STEP BY STEP.\\n"
        ),
        "user": "User's played games (game name and hours played; exclude all of these):\\n{history}"
    },
}

PROMPT_TEMPLATES_B ={
    "zero_shot": {
        "system": (
            "You are a helpful game recommender.\\n"
            "Recommend exactly the top-{k} ranked games.\\n"
            "RULES (STRICT):\\n"
            "- Recommend {k} distinct games not in the played list below.\\n"
            "- No duplicates.\\n"
            "- Output a single JSON object and nothing else.\\n"
            "SCHEMA (STRICT):\n"
            '{{\"k\": {k}, \"recommendations\": [\"<Game 1>\", \"<Game 2>\", ..., \"<Game {k}>\"]}}'
        ),
        "user": "User's played games (game name and hours played; exclude all of these):\\n{history}"
    },
    "few_shot": {
        "system": (
            "You are a helpful game recommender.\\n"
            "Below are demonstrated examples (no reasoning). Each example shows a user's played games (game name and hours played) and a FINAL_JSON.\\n"
            "Recommend exactly the top-{k} ranked games.\\n"
            "RULES (STRICT):\\n"
            "- Recommend {k} distinct games not in the played list below.\\n"
            "- No duplicates. Output a single JSON.\\n"
            "SCHEMA (STRICT):\n"
            '{{\"k\": {k}, \"recommendations\": [\"<Game 1>\", \"<Game 2>\", ..., \"<Game {k}>\"]}}'
        ),
        "user": "User's played games (game name and hours played; exclude all of these):\\n{history}"
    },
    "zero_shot_cot": {
        "system": (
            "You are a helpful game recommender.\\n"
            "Recommend exactly the top-{k} ranked games.\\n"
            "RULES (STRICT):\\n"
            "- Recommend {k} distinct games not in the played list below.\\n"
            "- No duplicates.\\n"
            "- Output a single JSON object and nothing else.\\n"
             "SCHEMA (STRICT):\n"
            '{{\"k\": {k}, \"recommendations\": [\"<Game 1>\", \"<Game 2>\", ..., \"<Game {k}>\"]}}'
            "Before answering, LET'S THINK STEP BY STEP.\\n"
        ),
        "user": "User's played games (game name and hours played; exclude all of these):\\n{history}"
    },
    "few_shot_cot": {
        "system": (
            "You are a helpful game recommender.\\n"
            "Below are demonstrated examples (no reasoning). Each example shows a user's played games (game name and hours played) and a FINAL_JSON.\\n"
            "Recommend exactly the top-{k} ranked games.\\n"
            "RULES (STRICT):\\n"
            "- Recommend {k} distinct games not in the played list below.\\n"
            "- No duplicates. Output a single JSON.\\n"
             "SCHEMA (STRICT):\n"
            '{{\"k\": {k}, \"recommendations\": [\"<Game 1>\", \"<Game 2>\", ..., \"<Game {k}>\"]}}'
            "Before answering, LET'S THINK STEP BY STEP.\\n"
        ),
        "user": "User's played games (game name and hours played; exclude all of these):\\n{history}"
    },
}

PROMPT_TEMPLATES_C = {
    "zero_shot": {
        "system": (
            "You are a helpful game recommender.\\n"
            "Recommend exactly the top-{k} ranked games.\\n"
            "RULES (STRICT):\\n"
            "- Recommend {k} distinct games not in the played list below.\\n"
            "- No duplicates.\\n"
            "SCHEMA (STRICT):\n"
            "RECOMMENDATIONS:\n1) <Game 1>\n2) <Game 2>\n...\n{k}) <Game {k}>"
        ),
        "user": "User's played games (game name and hours played; exclude all of these):\\n{history}"
    },
    "few_shot": {
        "system": (
            "You are a helpful game recommender.\\n"
            "Below are demonstrated examples (no reasoning). Each example shows a user's played games (game name and hours played) and a FINAL_JSON.\\n"
            "Recommend exactly the top-{k} ranked games.\\n"
            "RULES (STRICT):\\n"
            "- Recommend {k} distinct games not in the played list below.\\n"
            "- No duplicates.\\n"
            "SCHEMA (STRICT):\n"
            "RECOMMENDATIONS:\n1) <Game 1>\n2) <Game 2>\n...\n{k}) <Game {k}>"
        ),
        "user": "User's played games (game name and hours played; exclude all of these):\\n{history}"
    },
    "zero_shot_cot": {
        "system": (
            "You are a helpful game recommender.\\n"
            "Recommend exactly the top-{k} ranked games.\\n"
            "RULES (STRICT):\\n"
            "- Recommend {k} distinct games not in the played list below.\\n"
            "- No duplicates.\\n"
            "SCHEMA (STRICT):\n"
            "RECOMMENDATIONS:\n1) <Game 1>\n2) <Game 2>\n...\n{k}) <Game {k}>"
            "Before answering, LET'S THINK STEP BY STEP.\\n"
        ),
        "user": "User's played games (game name and hours played; exclude all of these):\\n{history}"
    },
    "few_shot_cot": {
        "system": (
            "You are a helpful game recommender.\\n"
            "Below are demonstrated examples (no reasoning). Each example shows a user's played games (game name and hours played) and a FINAL_JSON.\\n"
            "Recommend exactly the top-{k} ranked games.\\n"
            "RULES (STRICT):\\n"
            "- Recommend {k} distinct games not in the played list below.\\n"
            "SCHEMA (STRICT):\n"
            "RECOMMENDATIONS:\n1) <Game 1>\n2) <Game 2>\n...\n{k}) <Game {k}>"
            "Before answering, LET'S THINK STEP BY STEP.\\n"
        ),
        "user": "User's played games (game name and hours played; exclude all of these):\\n{history}"
    },
}

TEMPLATE_SETS = {"A": PROMPT_TEMPLATES_A, "B": PROMPT_TEMPLATES_B, "C": PROMPT_TEMPLATES_C}

## 4. Format history (games + hours) and build few-shot example turns (10 history + 10 recommendations)

**Note:** When few-shot examples are appended to the prompt, their **format matches the current template** (A, B, or C):
- Template A → examples use JSON with `{"rank": i, "title": "<game name>"}`.
- Template B → examples use JSON with `["<game name>", ...]` only.
- Template C → examples use plain text `RECOMMENDATIONS:\n1) <game name>\n...`.

In [ ]:
def format_history(pairs):
    """pairs = list of (game_title, hours)."""
    if not pairs:
        return ""
    lines = []
    for i, (game, hours) in enumerate(pairs, start=1):
        h = hours if isinstance(hours, (int, float)) else 0
        lines.append(f'{i}. "{game}" ({h} hours)')
    return "\n".join(lines)


def _history_text_from_example_steam(hist):
    """hist = list of {game, hours} from examples_steam.json"""
    if not hist:
        return "User's played games (game name and hours played; exclude all):\n"
    pairs = [(x.get("game", ""), x.get("hours", 0)) for x in hist]
    return "User's played games (game name and hours played; exclude all):\n" + format_history(pairs)


def _ranked_obj_from_titles(titles, k_default=10):
    titles = [str(t).strip() for t in titles if str(t).strip()][:k_default]
    items = [{"rank": i, "title": t} for i, t in enumerate(titles, start=1)]
    return {"k": len(items), "recommendations": items}


_REC_ITEM_RE = re.compile(r'(\d+)\.\s*(.+?)(?=(?:\s*,\s*\d+\.)|$)')

def recommendation_line_to_json_obj(rec_any, k_default=10):
    if isinstance(rec_any, dict) and "recommendations" in rec_any:
        items = []
        for i, it in enumerate(rec_any.get("recommendations", []), start=1):
            title = it.get("title", it.get("name", str(it))) if isinstance(it, dict) else str(it)
            items.append({"rank": i, "title": str(title).strip()})
        return {"k": len(items), "recommendations": items[:k_default]}
    if isinstance(rec_any, list):
        return _ranked_obj_from_titles(rec_any, k_default)
    rec_text = (rec_any or "").strip()
    if rec_text.startswith("{"):
        try:
            obj = json.loads(rec_text)
            if isinstance(obj, dict) and "recommendations" in obj:
                return recommendation_line_to_json_obj(obj, k_default)
        except Exception:
            pass
    items = []
    for m in _REC_ITEM_RE.finditer(rec_text):
        try:
            rank = int(m.group(1))
            title = m.group(2).strip().rstrip(", ").strip()
            items.append({"rank": rank, "title": title})
        except Exception:
            continue
    items = sorted(items, key=lambda x: x["rank"])[:k_default]
    return {"k": len(items), "recommendations": items} if items else {"k": 0, "recommendations": []}


def _assistant_text_for_set_A(rec_any, k_default=10):
    obj = recommendation_line_to_json_obj(rec_any, k_default)
    return json.dumps(obj, ensure_ascii=False, separators=(",", ":"))

def _assistant_text_for_set_B(rec_any, k_default=10):
    obj = recommendation_line_to_json_obj(rec_any, k_default)
    titles = [r["title"] for r in obj.get("recommendations", [])]
    return json.dumps({"k": len(titles), "recommendations": titles}, ensure_ascii=False, separators=(",", ":"))

def _assistant_text_for_set_C(rec_any, k_default=10):
    obj = recommendation_line_to_json_obj(rec_any, k_default)
    titles = [r["title"] for r in obj.get("recommendations", [])]
    return "RECOMMENDATIONS:\n" + "\n".join(f"{i}) {t}" for i, t in enumerate(titles, start=1))


def build_example_turns(set_name, k_default=10, max_examples=2, strategy="random"):
    """Build few-shot turns from examples_steam.json. Each example = 10 history (game name + hours) + 10 recommendations.
    IMPORTANT: The appended example format matches the current template (set_name):
    - A: assistant output = JSON with rank + title (game name); B: JSON with titles array only; C: plain text RECOMMENDATIONS:\n1) <game>\n..."""
    ex_list = EXAMPLES_DATA.get(strategy, [])
    if not ex_list:
        return []
    turns = []
    for ex in ex_list[:max_examples]:
        u_hist = _history_text_from_example_steam(ex.get("user_history", []))
        rec = ex.get("recommendation", [])
        if set_name == "A":
            a_out = _assistant_text_for_set_A(rec, k_default)
        elif set_name == "B":
            a_out = _assistant_text_for_set_B(rec, k_default)
        else:
            a_out = _assistant_text_for_set_C(rec, k_default)
        turns.append({"role": "user", "content": u_hist})
        turns.append({"role": "assistant", "content": a_out})
    return turns

## 5. Build messages and API call

In [ ]:
def _template_str(x):
    """CoT templates sometimes have 'system' as a tuple (comma in parentheses); normalize to string."""
    if isinstance(x, tuple):
        return "".join(str(t) for t in x)
    return str(x)

def render(text, **vals):
    text = _template_str(text)
    for k, v in vals.items():
        text = text.replace("{" + k + "}", str(v))
    return text


# Set how many tasks to print (prompt + response) in console; None = print all, 0 = none
PRINT_FIRST_N = 2


def print_messages_to_console(messages, max_chars=2500):
    """Print the prompt (messages) sent to the model in a readable way."""
    print("\n" + "=" * 80)
    print("PROMPT (messages sent to model)")
    print("=" * 80)
    for i, m in enumerate(messages):
        role = m.get("role", "?").upper()
        content = m.get("content", "")
        if len(content) > max_chars:
            content = content[:max_chars] + "... [truncated]"
        print(f"\n--- [{role}] ---\n{content}")
    print("=" * 80 + "\n")


def build_messages(set_name, style, k, real_history_text):
    tpl = TEMPLATE_SETS[set_name][style]
    system_text = _template_str(tpl["system"])
    user_tpl_text = _template_str(tpl["user"])
    messages = [{"role": "system", "content": render(system_text, k=k)}]
    if "few_shot" in style:
        # Few-shot examples are formatted to match this template (A, B, or C) so the model sees the same output format.
        messages.extend(build_example_turns(set_name, k_default=k, max_examples=2, strategy="random"))
    user_text = render(user_tpl_text, history=real_history_text, k=k)
    messages.append({"role": "user", "content": user_text})
    return messages


def chat_once(messages, temperature):
    for attempt in range(3):
        try:
            resp = client.chat.completions.create(model=MODEL_NAME, temperature=temperature, seed=42, messages=messages, max_tokens=500)
            answer = resp.choices[0].message.content.strip()
            return answer
        except Exception as e:
            print(f"Error: {e}, retry {attempt+1}/3")
            time.sleep(PAUSE_SECONDS)
    return "ERROR"

## 6. Run at temp=0 (all users × all prompt types × A, B, C)

In [ ]:
def _load_done_index_from_jsonl(path):
    done = set()
    if not path.exists():
        return done
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                key = (str(obj.get("user_id")), obj.get("template_set"), obj.get("style"), float(obj.get("temperature", 0.0)))
                done.add(key)
            except Exception:
                continue
    return done

def _load_done_count_from_jsonl(path):
    """Return Counter of (uid, set_name, style, temp) -> number of samples (for varying_temp with 20 samples each)."""
    from collections import Counter
    counts = Counter()
    if not path.exists():
        return counts
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                key = (str(obj.get("user_id")), obj.get("template_set"), obj.get("style"), float(obj.get("temperature", 0.0)))
                counts[key] += 1
            except Exception:
                continue
    return counts


def run_normal_t0(df_users, template_sets=("A", "B", "C")):
    out_path = OUTPUT_NORMAL / "normal_t0.jsonl"
    done = _load_done_index_from_jsonl(out_path)
    tasks = []
    for _, row in df_users.iterrows():
        uid = str(row.user_id)
        history_text = format_history(row.get("sample_random", []))
        for set_name in template_sets:
            for style in TEMPLATE_SETS[set_name].keys():
                if (uid, set_name, style, 0.0) in done:
                    continue
                tasks.append((uid, set_name, style, history_text))
    if not tasks:
        print("All T=0 tasks already done.")
        return
    print_count = [0]  # mutable so lambda can update
    write_lock = Lock()
    with out_path.open("a", encoding="utf-8") as f_out:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
            futures = []
            for u, sn, st, ht in tasks:
                def _task(u=u, sn=sn, st=st, ht=ht):
                    msgs = build_messages(sn, st, TOP_K, ht)
                    answer = chat_once(msgs, 0.0)
                    return (u, sn, st, 0.0, answer, msgs)
                futures.append(pool.submit(_task))
            for fut in tqdm(as_completed(futures), total=len(futures), desc="Normal T=0"):
                try:
                    user_id, set_name, style, temp, answer, msgs = fut.result()
                except Exception as e:
                    print(f"Future failed: {e}")
                    continue
                # Print prompt and model response to console for the first N *successful* tasks
                if PRINT_FIRST_N is not None and PRINT_FIRST_N > 0:
                    print_count[0] += 1
                    if print_count[0] <= PRINT_FIRST_N:
                        print(f"\n>> Task {print_count[0]} (user={user_id}, template={set_name}, style={style})")
                        print_messages_to_console(msgs)
                        print("MODEL RESPONSE:\n", answer)
                        print("\n")
                record = {"user_id": user_id, "template_set": set_name, "style": style, "temperature": temp, "messages": msgs, "response": answer}
                with write_lock:
                    f_out.write(json.dumps(record, ensure_ascii=False) + "\n")
                    f_out.flush()
    print(f"Saved to {out_path}")

In [ ]:
# Reproducibility: run steam_data_prep.py first (uses seed 42). User selection below uses seed 42.
random.seed(42)
picked = sorted(random.sample(list(df_filtered_user_data["user_id"]), min(100, len(df_filtered_user_data))))
users_to_run = df_filtered_user_data[df_filtered_user_data["user_id"].isin(picked)].copy()
print(f"Running for {len(users_to_run)} users @ T=0, all prompt types and templates A,B,C")
run_normal_t0(users_to_run, template_sets=("A", "B", "C"))

## 7. Run at varying temperatures (optional)

In [ ]:
def run_varying_temp(df_users, temps=None, template_sets=("A", "B", "C"), num_samples_cot=None, styles_20=None):
    """Run at varying temps: all 4 prompt types. zero_shot/few_shot: 1 sample each; zero_shot_cot/few_shot_cot: 20 samples (self-consistency)."""
    temps = temps or VARIING_TEMPS
    num_samples_cot = num_samples_cot if num_samples_cot is not None else NUM_SAMPLES_VARYING_TEMP
    styles_20 = styles_20 if styles_20 is not None else STYLES_20_SAMPLES
    out_path = OUTPUT_NORMAL / "varying_temp.jsonl"
    done_counts = _load_done_count_from_jsonl(out_path)
    tasks = []
    for _, row in df_users.iterrows():
        uid = str(row.user_id)
        history_text = format_history(row.get("sample_random", []))
        for set_name in template_sets:
            for style in TEMPLATE_SETS[set_name].keys():
                n = num_samples_cot if style in styles_20 else 1
                for t in temps:
                    key = (uid, set_name, style, t)
                    have = done_counts.get(key, 0)
                    for sample_idx in range(have, n):
                        tasks.append((uid, set_name, style, history_text, t, sample_idx))
    if not tasks:
        print("All varying-temp tasks done (1 sample for zero_shot/few_shot, 20 for zero_shot_cot/few_shot_cot).")
        return
    write_lock = Lock()
    with out_path.open("a", encoding="utf-8") as f_out:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS_VARYING_TEMP) as pool:
            futures = []
            for uid, set_name, style, ht, t, sample_idx in tasks:
                def _task(u=uid, sn=set_name, st=style, h=ht, temp=t, idx=sample_idx):
                    msgs = build_messages(sn, st, TOP_K, h)
                    answer = chat_once(msgs, temp)
                    return (u, sn, st, temp, idx, msgs, answer)
                futures.append(pool.submit(_task))
            for fut in tqdm(as_completed(futures), total=len(futures), desc="Varying temp"):
                try:
                    uid, set_name, style, t, sample_idx, msgs, answer = fut.result()
                except Exception as e:
                    print(f"Future failed: {e}")
                    continue
                record = {"user_id": uid, "template_set": set_name, "style": style, "temperature": t, "sample_idx": sample_idx, "messages": msgs, "response": answer}
                with write_lock:
                    f_out.write(json.dumps(record, ensure_ascii=False) + "\n")
                    f_out.flush()
    print(f"Saved to {out_path}")

# Varying temp: all 4 prompt types (zero_shot, few_shot, zero_shot_cot, few_shot_cot) at temps [0.2,0.5,0.7,1.0,1.2,1.4,1.6].
# zero_shot/few_shot: 1 sample each; zero_shot_cot/few_shot_cot: 20 samples each (self-consistency). Template B only.
run_varying_temp(users_to_run, temps=VARIING_TEMPS, template_sets=("B",), num_samples_cot=NUM_SAMPLES_VARYING_TEMP, styles_20=STYLES_20_SAMPLES)

## 8. Self-consistent CoT (run CoT at varying temps, aggregate e.g. majority vote)

In [ ]:
# varying_temp.jsonl (template B): zero_shot/few_shot have 1 sample each; zero_shot_cot/few_shot_cot have 20 samples each (self-consistency).